# Финансовый навигатор — решение (CatBoost, multi-label)

Мульти-лейбл классификация: для каждого клиента — вероятность интереса к 9 продуктам.
Метрика: macro ROC-AUC. Модель: 9 x CatBoost (one-vs-rest) с 5-fold bagging.

**Честная локальная оценка: MACRO OOF AUC ~ 0.652.**

Пайплайн воспроизводит то, что собрано в Docker-образе `flexonafft/ci-navigator:1.0`:
`train.py` обучает модели локально, `run.py` внутри контейнера усредняет фолды и пишет `output.csv`.

## 1. Импорты и константы

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

PRODUCTS = [
    "credit_card", "mortgage", "deposit", "investment", "insurance",
    "p2p_transfer", "cashback", "premium_account", "business_loan",
]
TARGETS = [f"product_{p}" for p in PRODUCTS]

FEATURES = [
    "age", "income_bucket", "tenure_months", "tx_count_30d",
    "avg_tx_amount", "digital_activity_score", "has_child", "is_salary_client",
]
CAT_FEATURES = ["income_bucket", "has_child", "is_salary_client"]

N_FOLDS = 5
SEED = 42

## 2. Данные и быстрый EDA
Проверяем баланс таргетов и где есть сигнал.

In [ ]:
df = pd.read_csv("../train_weRmhWx.csv")
print("shape:", df.shape)
print("\nДоля положительных по продуктам:")
print(df[TARGETS].mean().round(3))

d = df.copy()
d["has_child"] = d["has_child"].astype(int)
d["is_salary_client"] = d["is_salary_client"].astype(int)
corr = d[FEATURES + TARGETS].corr().loc[FEATURES, TARGETS].abs()
print("\nМакс |corr| признак->таргет по каждому признаку:")
print(corr.max(axis=1).round(3))

**Вывод EDA:** сигнал сосредоточен в `income_bucket`, `digital_activity_score`, `has_child`, `is_salary_client`
и нелинеен (например `has_child`->ипотека/страховка, `income_bucket`->премиум/бизнес-кредит).
Признаки `age/tenure_months/tx_count_30d/avg_tx_amount` близки к шуму — CatBoost их регуляризует сам.
Поэтому градиентный бустинг с нативной обработкой категориальных подходит лучше всего.

## 3. Подготовка признаков

In [ ]:
def make_features(frame: pd.DataFrame) -> pd.DataFrame:
    X = frame[FEATURES].copy()
    X["has_child"] = X["has_child"].astype(int)
    X["is_salary_client"] = X["is_salary_client"].astype(int)
    return X

X = make_features(df)
Y = df[TARGETS].values
cat_idx = [FEATURES.index(c) for c in CAT_FEATURES]

## 4. Обучение: 9 x CatBoost, 5-fold bagging
На каждый продукт — 5 моделей по фолдам; OOF даёт честную оценку. Модели сохраняются в `./models/*.cbm`.

In [ ]:
PARAMS = dict(
    iterations=1500, learning_rate=0.03, depth=6, l2_leaf_reg=5.0,
    loss_function="Logloss", eval_metric="AUC", random_seed=SEED, verbose=0,
)

out = Path("./models"); out.mkdir(exist_ok=True)
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
oof_aucs = {}

for j, prod in enumerate(PRODUCTS):
    y = Y[:, j]
    oof = np.zeros(len(df))
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = CatBoostClassifier(cat_features=cat_idx, early_stopping_rounds=80, **PARAMS)
        model.fit(X.iloc[tr], y[tr], eval_set=(X.iloc[va], y[va]))
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]
        model.save_model(str(out / f"{prod}_fold{fold}.cbm"))
    oof_aucs[prod] = roc_auc_score(y, oof)
    print(f"{prod:24s} OOF AUC = {oof_aucs[prod]:.5f}")

macro = float(np.mean(list(oof_aucs.values())))
print(f"\nMACRO OOF AUC = {macro:.5f}")

config = {"products": PRODUCTS, "features": FEATURES, "cat_features": CAT_FEATURES,
          "n_folds": N_FOLDS, "oof_auc": oof_aucs, "macro_oof_auc": macro}
(out / "config.json").write_text(json.dumps(config, indent=2))

## 5. Инференс (как в `run.py` внутри контейнера)

Контейнер получает `--input-path` / `--output-path`. Для каждого продукта вероятности
усредняются по 5 фолдам. Результат — CSV без header, 9 колонок в порядке `PRODUCTS`.

In [ ]:
def predict(input_path, output_path, models_dir="./models"):
    cfg = json.loads(Path(models_dir, "config.json").read_text())
    test = pd.read_csv(input_path)
    Xt = make_features(test)
    preds = np.zeros((len(test), len(cfg["products"])))
    for j, prod in enumerate(cfg["products"]):
        acc = np.zeros(len(test))
        for fold in range(cfg["n_folds"]):
            m = CatBoostClassifier(); m.load_model(str(Path(models_dir, f"{prod}_fold{fold}.cbm")))
            acc += m.predict_proba(Xt)[:, 1]
        preds[:, j] = acc / cfg["n_folds"]
    pd.DataFrame(preds).to_csv(output_path, header=False, index=False)
    return preds

df.head(200).to_csv("/tmp/_demo_in.csv", index=False)
p = predict("/tmp/_demo_in.csv", "/tmp/_demo_out.csv")
print("output shape:", p.shape, "| диапазон:", round(p.min(),4), "..", round(p.max(),4))

## 6. Воспроизводимость / деплой

Образ собирается под `linux/amd64` и пушится на Docker Hub:

```bash
docker build --platform linux/amd64 -t flexonafft/ci-navigator:1.0 .
docker push flexonafft/ci-navigator:1.0
```

Зависимости (`requirements.txt`): `catboost==1.2.8`, `pandas==2.2.3`, `numpy==1.26.4`, `scipy==1.13.1`.